# Learn the Librarian by running it

This notebook is a guided tour of Linger's Librarian. You do not need to know search systems, machine learning, or the Linger codebase before you begin.

By the end, you will be able to:

- ask the Librarian a question about *Alice's Adventures in Wonderland*;
- control which chapters it may search, so it cannot reveal later spoilers;
- see how passages are found and reordered; and
- understand the scores used to evaluate the result.

> **Cost note:** Search runs locally. The cells labelled **AI call** use the model and API key configured in `.env`. The optional all-case evaluation can make several AI calls.

## The big picture

The Librarian receives a question and a reading position. It searches only the chapters the reader is allowed to see.

```text
Reader's question
       ↓
Confirm the spoiler boundary
       ↓
Keyword search + meaning-based search
       ↓
Combine duplicates and rerank passages
       ↓
Decide: sufficient, weak, or no evidence
       ↓
Return exact passages to Muse
```

After the shared setup, choose either route. They are independent:

| Route | Use it when you want to… | What you see |
|---|---|---|
| **A — End-to-end run** | Ask your own question and use the Librarian as Muse would | The final clarification, failure, or evidence result |
| **B — Step-by-step walkthrough** | Learn or debug how retrieval works internally | Boundary filtering, both searches, fusion, reranking, and evaluation |

> **Route B does not require Route A.** Route A is not a prerequisite or an earlier stage of Route B. They are two different views of the Librarian.

The notebook only displays results. It does not overwrite the checked-in evaluation reports.

## Before you run anything

1. Open the **whole `linger` repository folder** in VS Code, not only this notebook file.
2. Select `.venv/bin/python` as the notebook kernel using **Select Kernel** in the upper-right corner.
3. Make sure `.env` contains `LINGER_MODEL` and the matching provider API key.
4. Run the **Shared setup** cells with the ▶ button or `Shift+Enter`.
5. Choose Route A or Route B. Within the route you choose, run its cells from top to bottom. You do not need to run the other route.

> Cells marked **Edit this cell** contain safe inputs for you to change. Cells marked **Run without editing** are setup or helper code.

## Small glossary

| Term | Plain-language meaning |
|---|---|
| **Corpus** | The collection of books the Librarian is allowed to search. This notebook uses one version of *Alice in Wonderland*. |
| **Window / passage** | A small piece of a chapter used as one search candidate. A window never crosses a chapter boundary. |
| **BM25 / keyword search** | Finds passages containing important words from the question. |
| **Semantic search** | Finds passages with a similar meaning, even when the wording differs. |
| **Fusion** | Combines the keyword and semantic result lists and removes repeated passages. |
| **Reranking** | Reads the candidates more carefully and puts the most relevant ones first. |
| **Evidence strength** | The Librarian's final judgment: the passages are `sufficient`, `weak`, or `none` for answering the question. |
| **Gold evidence** | A passage a human marked as the expected answer for an evaluation example. The Librarian never sees this label while searching. |
| **Spoiler boundary** | The latest chapter the search is allowed to inspect. |

# Shared setup — Run once before either route

**Run without editing.** The next two cells locate the repository, load the required code, and check the model configuration. They print only whether an API key exists—not the key itself.

**Success looks like:** `catalog_exists` and `provider_key_configured` are both `true`.

In [ ]:
import html
import os
from pathlib import Path
import sys
from time import perf_counter

import polars as pl
from IPython.display import HTML, JSON, Markdown, display

def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "apps").is_dir():
            return candidate
    raise RuntimeError(
        "Could not find the Linger repository. Start JupyterLab from the repository "
        "root or its notebooks directory."
    )

ROOT = find_repository_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("LOGFIRE_IGNORE_NO_CONFIG", "1")

from apps.backend.config import get_settings
from evals.librarian.benchmark import (
    STRATEGIES, BenchmarkRunner, Hit, load_cases, load_windows, measure,
)
from src.linger.agents.muse.tools import librarian_search
from src.linger.contracts.librarian import (
    ClarificationRequest, RetrievalFailure, RetrievalResult,
)
from src.linger.contracts.reading import ReadingBoundary
from src.linger.contracts.turn import ConfirmedReading
from src.linger.corpus.alice import BOOK, BOOK_VERSION_ID, WORK_ID
from src.linger.orchestration.turn_context import (
    reset_confirmed_reading, set_confirmed_reading,
)

def show_librarian_response(response):
    """Show a readable summary first and keep technical details optional."""
    if isinstance(response, ClarificationRequest):
        accepted = (
            ", ".join(response.expected_answer.values)
            if response.expected_answer.values
            else "Free-text answer"
        )
        display(Markdown("### 🟡 Clarification needed"))
        display(pl.DataFrame([{
            "Question to ask the reader": response.question,
            "Why search stopped": response.reason_code,
            "Accepted answer": accepted,
        }]))
    elif isinstance(response, RetrievalFailure):
        retry_text = "Yes" if response.retryable else "No"
        display(Markdown("### 🔴 Librarian could not finish safely"))
        display(pl.DataFrame([{
            "Error": response.error_code,
            "Safe to retry": retry_text,
            "Evidence returned": 0,
        }]))
    else:
        icon = {"sufficient": "✅", "weak": "🟠", "none": "⚪"}[
            response.evidence_strength
        ]
        limitations = (
            "\n".join(f"- {item}" for item in response.limitations)
            if response.limitations
            else "- None reported"
        )
        display(Markdown(
            f"### {icon} Search completed: {response.evidence_strength} evidence\n\n"
            f"**Why:** {response.strength_reason}\n\n"
            f"**Searched:** Chapters 1–{response.searched_scope.max_chapter_inclusive} "
            f"of `{response.searched_scope.book_version_id}`\n\n"
            f"**Final passages returned:** {len(response.evidence)}\n\n"
            f"**Limitations:**\n{limitations}"
        ))
        if response.evidence:
            rows = []
            full_passages = []
            for rank, record in enumerate(response.evidence, start=1):
                preview = " ".join(record.text.split())
                if len(preview) > 220:
                    preview = preview[:217] + "..."
                rows.append({
                    "Rank": rank,
                    "Chapter": record.chapter_number,
                    "Source lines": f"{record.source_lines[0]}–{record.source_lines[1]}",
                    "Evidence ID": record.evidence_id,
                    "Preview": preview,
                })
                full_passages.append(
                    "<details style='margin:.6rem 0'>"
                    f"<summary><b>Passage {rank}: {html.escape(record.location)}</b></summary>"
                    f"<pre style='white-space:pre-wrap'>{html.escape(record.text)}</pre>"
                    "</details>"
                )
            display(pl.DataFrame(rows))
            display(HTML("".join(full_passages)))
        else:
            display(Markdown("*No passages were returned inside the allowed scope.*"))

    raw_json = html.escape(response.model_dump_json(indent=2))
    display(HTML(
        "<details style='margin-top:1rem'>"
        "<summary><b>Show technical JSON</b></summary>"
        f"<pre style='white-space:pre-wrap'>{raw_json}</pre>"
        "</details>"
    ))
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(12)

In [ ]:
settings = get_settings()
provider_name = settings.linger_model.partition(":")[0]
try:
    settings.api_key_for(provider_name)
except RuntimeError:
    provider_key_configured = False
else:
    provider_key_configured = True
setup = {
    "repository_root": str(ROOT),
    "model": settings.linger_model,
    "provider_key_configured": provider_key_configured,
    "book_version_id": BOOK_VERSION_ID,
    "catalog_exists": (BOOK.default_output / "catalog.json").is_file(),
}
display(JSON(setup))
if provider_key_configured and setup["catalog_exists"]:
    display(Markdown("✅ **Setup is ready.** Continue to Part 1."))
elif not provider_key_configured:
    display(Markdown(
        f"⚠️ Add the API key for `{provider_name}` to `.env` before a full Librarian run. "
        "The local retrieval evaluation can still run without it."
    ))
elif not setup["catalog_exists"]:
    display(Markdown(
        "⚠️ The Alice corpus catalog is missing. Run the corpus check described in the root README."
    ))

# Route A — End-to-end Librarian run

Use this route when you want to treat the Librarian as one complete component: provide a question and reading boundary, then inspect the final response. This route does **not** expose every internal search stage.

## A1 — Choose the question and reading position

**Edit this cell.** These are the only values you need to change for a normal manual run:

| Setting | What it controls | Example |
|---|---|---|
| `QUERY` | What you want to find in the book | `Why is Alice confused?` |
| `CURRENT_CHAPTER` | The chapter the reader is currently on | `5` |
| `CHAPTER_STATE` | Whether the current chapter is finished | `"completed"` or `"started"` |
| `MAX_FINAL_EVIDENCE` | Maximum passages returned to Muse | `5` |

The spoiler rule is important:

- `"completed"` at Chapter 5 searches Chapters 1–5.
- `"started"` at Chapter 5 searches Chapters 1–4.
- `None` means the boundary is unclear, so the Librarian asks a clarification question and does not search.

### Edit the next cell

Change the four values, then run the cell to confirm the effective chapter ceiling before any search occurs.

In [ ]:
# EDIT THESE FOUR VALUES.
QUERY = "Why can Alice not explain who she is to the Caterpillar?"
CURRENT_CHAPTER = 5
CHAPTER_STATE = "completed"  # "completed", "started", or None
MAX_FINAL_EVIDENCE = 5

if CHAPTER_STATE not in {"completed", "started", None}:
    raise ValueError("CHAPTER_STATE must be 'completed', 'started', or None")
effective_ceiling = (
    None if CHAPTER_STATE is None
    else CURRENT_CHAPTER - int(CHAPTER_STATE == "started")
)
display(JSON({
    "query": QUERY,
    "current_chapter": CURRENT_CHAPTER,
    "chapter_state": CHAPTER_STATE,
    "effective_search_ceiling": effective_ceiling,
}))

## A2 — Load the end-to-end helper

**Run without editing.** This helper safely sets the reading boundary, calls the Librarian, and then clears the temporary reading context. Defining it does not make an AI call.

In [ ]:
async def run_librarian(query, current_chapter, chapter_state, max_final_evidence=5):
    boundary = (
        None if chapter_state is None
        else ReadingBoundary(
            chapter_number=current_chapter, chapter_state=chapter_state,
        )
    )
    token = set_confirmed_reading(
        ConfirmedReading(work_id=WORK_ID, chapter_max=current_chapter)
    )
    try:
        return await librarian_search(
            query=query,
            work_id=WORK_ID,
            book_version_id=BOOK_VERSION_ID,
            reading_boundary=boundary,
            max_final_evidence=max_final_evidence,
        )
    finally:
        reset_confirmed_reading(token)

## A3 — Run the Librarian end to end

**AI call when retrieval finds candidates.** Run the next cell without editing. It prints the exact typed response and the total elapsed time. The first run is slower because local search models may need to download and initialize.

In [ ]:
started = perf_counter()
manual_response = await run_librarian(
    QUERY, CURRENT_CHAPTER, CHAPTER_STATE, MAX_FINAL_EVIDENCE,
)
manual_latency_ms = (perf_counter() - started) * 1_000
show_librarian_response(manual_response)
display(Markdown(f"**Total elapsed time:** {manual_latency_ms:,.1f} ms"))

### How to read the response

Look first at `kind`:

| `kind` | Meaning | What happens next |
|---|---|---|
| `clarification` | The book or spoiler boundary is unclear. Search did **not** run. | Muse asks the returned `question`. |
| `failure` | A dependency failed, so the Librarian stopped safely. | Check `error_code`; retry only when `retryable` is `true`. |
| `result` | Search completed. | Read `evidence_strength`, `strength_reason`, and `evidence`. |

For a `result`:

- `sufficient` means the returned passages can directly support a useful answer.
- `weak` means the passages help but do not fully answer the question; check `limitations`.
- `none` means no useful evidence was found inside the permitted chapters. This is a completed search, not a system failure.
- `searched_scope.max_chapter_inclusive` shows the highest chapter that was actually searched.
- Each `evidence_id` points to exact lines in the immutable book source.

# Route B — Step-by-step internal walkthrough

Use this route when you want to understand, debug, or evaluate the internal retrieval stages. It uses a frozen example with human-marked expected evidence. **You can start here after Shared setup; Route A is not required.**

## B1a — Browse the evaluation examples

An evaluation case contains a question, a spoiler boundary, the expected evidence-strength label, and human-marked answer lines. Those answer lines are called **gold evidence**. They are used only to score the result after searching—the Librarian never sees them.

First, run the next cell without editing to browse the available cases. It only shows the catalog; it does not select a case.

A good first example is `identity-change` because it has a clear answer in Chapter 5.

In [ ]:
benchmark = load_cases()
display(pl.DataFrame([{
    "Case ID": item.case_id,
    "Question": item.query,
    "Chapter ceiling": item.chapter_max,
    "Expected strength": item.expected_strength,
} for item in benchmark.cases]))

### B1b — Select one case

**Edit `CASE_ID` in the next cell**, then run it. Unlike the browser above, this cell displays exactly one selected row.

**Success looks like:** the value in **Selected case ID** exactly matches the `CASE_ID` you entered.

In [ ]:
# EDIT THIS ONE VALUE.
CASE_ID = "identity-change"
case = next((item for item in benchmark.cases if item.case_id == CASE_ID), None)
if case is None:
    raise ValueError(f"Unknown CASE_ID: {CASE_ID}")
gold_locations = ", ".join(
    f"Chapter {chapter}, lines {start}–{end}"
    for chapter, start, end in case.relevant_ranges
) or "No gold passage (expected result: no evidence)"
display(pl.DataFrame([{
    "Selected case ID": case.case_id,
    "Question": case.query,
    "Search through chapter": case.chapter_max,
    "Expected strength": case.expected_strength,
    "Human-marked gold locations": gold_locations,
}]))

## B2 — Prepare the searchable passages

**Run without editing.** The book is divided into overlapping passages of about 350 words. The overlap is about 60 words so an important sentence near the edge of one passage also appears in its neighbour. Passages never cross chapter boundaries.

This cell also loads the local meaning-search and reranking models. It may take a while the first time.

**Success looks like:** a positive `window_count` and an initialization time.

In [ ]:
all_windows = load_windows()
started = perf_counter()
runner = BenchmarkRunner(all_windows)
index_build_ms = (perf_counter() - started) * 1_000
display(JSON({
    "window_count": len(all_windows),
    "index_and_model_initialization_ms": round(index_build_ms, 1),
}))

def hit_table(hits):
    return pl.DataFrame([{
        "rank": rank,
        "evidence_id": hit.evidence_id,
        "chapter": hit.chapter,
        "source_lines": f"{hit.source_lines[0]}-{hit.source_lines[1]}",
        "score": round(hit.score, 4),
        "preview": " ".join(hit.text.split())[:180],
    } for rank, hit in enumerate(hits, start=1)])

## B3 — Prove the spoiler boundary is applied first

**Run without editing.** Before any search model sees the book, the application removes passages from later chapters. This is safer than searching the whole book and hiding spoilers afterward.

**Success looks like:** `boundary_pass` is `true`, and `max_chapter_entering_search` is not greater than `chapter_ceiling`.

In [ ]:
eligible_windows, _ = runner._eligible(case.chapter_max)
max_eligible_chapter = max(window.chapter for window in eligible_windows)
boundary_pass = max_eligible_chapter <= case.chapter_max
assert boundary_pass
display(JSON({
    "chapter_ceiling": case.chapter_max,
    "eligible_window_count": len(eligible_windows),
    "max_chapter_entering_search": max_eligible_chapter,
    "boundary_pass": boundary_pass,
}))

## B4 — Search in two complementary ways

**Run without editing.**

- **BM25** looks for important words shared by the question and passage. It is strong when the book uses similar wording.
- **Semantic search** compares meaning. It can find a useful passage even when it uses different words.

Each table shows rank, stable evidence ID, chapter, exact source lines, score, and a short preview. Scores from BM25 and semantic search are on different scales, so compare rank within a table—not raw scores across the two tables.

In [ ]:
keyword_hits = runner._bm25_hits(case.query, case.chapter_max)
semantic_hits = runner._semantic_hits(case.query, case.chapter_max)
display(Markdown(f"**BM25 candidates:** {len(keyword_hits)}"))
display(hit_table(keyword_hits))
display(Markdown(f"**Semantic candidates at score ≥ 0.5:** {len(semantic_hits)}"))
display(hit_table(semantic_hits))

## B5 — Combine the two result lists

**Run without editing.** Fusion rewards passages that rank well in either search method. Deduplication then removes strongly overlapping passages, so the next stage does not waste time reading the same text twice.

The fusion score is used only to order candidates for the next stage. It is not a probability that the passage answers the question.

In [ ]:
fused_hits = runner._fuse(keyword_hits, semantic_hits)
display(Markdown(f"**Fused candidates:** {len(fused_hits)}"))
display(hit_table(fused_hits))

## B6 — Rerank the strongest candidates

**Run without editing.** A local cross-encoder reads the question together with each candidate and produces a more careful relevance order. Candidates below 0.5 are removed, duplicates are checked again, and no more than five passages remain.

A high reranker score means *this passage resembles the question*. It still does not prove that the complete evidence can answer the question—that decision happens later.

In [ ]:
reranked_hits = runner.search("hybrid_reranked", case)
display(Markdown(f"**Final retrieval candidates:** {len(reranked_hits)}"))
display(hit_table(reranked_hits))

## B7 — Score the retrieved passages

**Run without editing.** The first output scores the selected hybrid-and-reranked result against the human-marked gold lines. After that, a separate comparison cell runs the same case through all five retrieval methods.

| Metric | Beginner-friendly question | Best value |
|---|---|---|
| Evidence recall | Did search find the required answer passage? | `1.0` |
| Candidate precision | How many returned candidates overlap human-marked useful passages? | `1.0` |
| Spoiler safe | Did every passage stay within the chapter ceiling? | `true` |
| Citations resolve | Does each evidence ID point to the exact canonical source text? | `true` |
| Latency | How long did this already-initialized search take? | Lower is faster |

Candidate precision is only a diagnostic. The Librarian may return a smaller and cleaner evidence set after judging answerability.

In [ ]:
source_lines = BOOK.default_source.read_text(encoding="utf-8").splitlines()
candidate_measurement = measure(case, reranked_hits, source_lines)
display(JSON({
    "evidence_recall": candidate_measurement.recall,
    "candidate_precision": candidate_measurement.precision,
    "strength_support_correct": candidate_measurement.strength_correct,
    "spoiler_safe": not candidate_measurement.forbidden_exposure,
    "citations_resolve": candidate_measurement.citations_resolve,
    "evidence_word_count": candidate_measurement.evidence_tokens,
}))

### Compare the five retrieval methods

**Run without editing.** This is a small ablation study: it keeps the question and spoiler boundary fixed, changes only the retrieval method, and shows what each method returns.

The methods build on one another:

1. `direct` — simple bounded chapter reading, used as the control;
2. `bm25` — keyword search;
3. `semantic` — meaning-based search;
4. `hybrid` — keyword and semantic results combined;
5. `hybrid_reranked` — the combined candidates reordered by the cross-encoder.

The timing here is a quick, warm one-case comparison. Use the aggregate benchmark—not this table—for a production performance decision.

In [ ]:
strategy_rows = []
for strategy in STRATEGIES:
    started = perf_counter()
    hits = runner.search(strategy, case)
    latency_ms = (perf_counter() - started) * 1_000
    measured = measure(case, hits, source_lines)
    strategy_rows.append({
        "strategy": strategy,
        "evidence_count": len(hits),
        "recall": measured.recall,
        "candidate_precision": measured.precision,
        "strength_support_correct": measured.strength_correct,
        "spoiler_safe": not measured.forbidden_exposure,
        "citations_resolve": measured.citations_resolve,
        "latency_ms": round(latency_ms, 3),
    })
display(pl.DataFrame(strategy_rows))

## B8 — Evaluate the final Librarian decision

The next helper cell defines how to compare the final response with the expected answer. **Run the helper without editing; it does not make an AI call.**

The following cell is an **AI call**: it runs production retrieval plus the configured evidence-strength model. The returned passages are the final evidence exposed to Muse.

**Success looks like:** `response_kind` is `result`, `strength_correct`, `spoiler_safe`, and `citations_resolve` are `true`, with recall and final evidence precision as close to `1.0` as possible.

In [ ]:
async def run_evaluation_case(evaluation_case):
    """Run one Route B case without depending on Route A."""
    token = set_confirmed_reading(
        ConfirmedReading(work_id=WORK_ID, chapter_max=evaluation_case.chapter_max)
    )
    try:
        return await librarian_search(
            query=evaluation_case.query,
            work_id=WORK_ID,
            book_version_id=BOOK_VERSION_ID,
            reading_boundary=ReadingBoundary(
                chapter_number=evaluation_case.chapter_max,
                chapter_state="completed",
            ),
            max_final_evidence=5,
        )
    finally:
        reset_confirmed_reading(token)

def evaluate_librarian_response(case, response):
    if not isinstance(response, RetrievalResult):
        return {
            "case_id": case.case_id,
            "response_kind": response.kind,
            "expected_strength": case.expected_strength,
            "predicted_strength": None,
            "strength_correct": False,
            "evidence_recall": None,
            "final_evidence_precision": None,
            "spoiler_safe": None,
            "citations_resolve": None,
        }
    hits = [Hit(
        evidence_id=record.evidence_id,
        chapter=record.chapter_number,
        source_lines=record.source_lines,
        text=record.text,
        score=1.0,
    ) for record in response.evidence]
    measured = measure(case, hits, source_lines)
    return {
        "case_id": case.case_id,
        "response_kind": response.kind,
        "expected_strength": case.expected_strength,
        "predicted_strength": response.evidence_strength,
        "strength_correct": response.evidence_strength == case.expected_strength,
        "evidence_recall": measured.recall,
        "final_evidence_precision": measured.precision,
        "spoiler_safe": not measured.forbidden_exposure,
        "citations_resolve": measured.citations_resolve,
    }

### Run and score the complete case

**AI call.** Run without editing. First you will see the Librarian's raw typed response. Under it, a one-row table compares the final result with the human expectation.

In [ ]:
started = perf_counter()
case_response = await run_evaluation_case(case)
case_latency_ms = (perf_counter() - started) * 1_000
show_librarian_response(case_response)
case_result = evaluate_librarian_response(case, case_response)
case_result["latency_ms"] = round(case_latency_ms, 1)
display(pl.DataFrame([case_result]))

# Route B optional exercise — Run every evaluation case

You can stop here if you only wanted to learn the flow.

The next cell runs all 12 frozen examples. It can make up to one evidence-strength AI call per case, so it takes longer and may use provider credits. Results stay in notebook memory and do not overwrite checked-in reports.

To run it, change only `RUN_ALL_CASES = False` to `RUN_ALL_CASES = True`. The final summary reports:

- the fraction of correct strength labels;
- average evidence recall and final evidence precision;
- whether every case stayed spoiler-safe; and
- whether every final citation resolved exactly.

In [ ]:
# Leave False for a single-case tutorial. Change to True only intentionally.
RUN_ALL_CASES = False

if RUN_ALL_CASES:
    all_case_rows = []
    for index, evaluation_case in enumerate(benchmark.cases, start=1):
        print(f"[{index}/{len(benchmark.cases)}] {evaluation_case.case_id}")
        started = perf_counter()
        response = await run_evaluation_case(evaluation_case)
        row = evaluate_librarian_response(evaluation_case, response)
        row["latency_ms"] = round((perf_counter() - started) * 1_000, 1)
        all_case_rows.append(row)
    display(pl.DataFrame(all_case_rows))
    result_rows = [r for r in all_case_rows if r["response_kind"] == "result"]
    summary = {
        "case_count": len(all_case_rows),
        "result_count": len(result_rows),
        "strength_accuracy": sum(r["strength_correct"] for r in all_case_rows) / len(all_case_rows),
        "mean_evidence_recall": None,
        "mean_final_evidence_precision": None,
        "all_spoiler_safe": None,
        "all_citations_resolve": None,
    }
    if result_rows:
        summary.update({
            "mean_evidence_recall": sum(r["evidence_recall"] for r in result_rows) / len(result_rows),
            "mean_final_evidence_precision": sum(r["final_evidence_precision"] for r in result_rows) / len(result_rows),
            "all_spoiler_safe": all(r["spoiler_safe"] for r in result_rows),
            "all_citations_resolve": all(r["citations_resolve"] for r in result_rows),
        })
    display(JSON(summary))
else:
    display(Markdown("Set `RUN_ALL_CASES = True` to run all credential-backed cases."))

# Final checklist

A healthy run has:

- no evidence above the chapter ceiling;
- every evidence ID resolving to exact book lines;
- the expected `sufficient`, `weak`, or `none` label; and
- high recall and precision on the frozen examples.

A lower score on one case does not automatically mean the system is unsafe. Spoiler safety and citation resolution are hard requirements; recall, precision, latency, and strength accuracy are quality measures used to compare approaches.

## Troubleshooting

| Problem | What to do |
|---|---|
| `ModuleNotFoundError` | In VS Code, choose the repository's `.venv/bin/python` with **Select Kernel**, restart the kernel, and run from the first cell. |
| `provider_key_configured` is `false` | Add the API key matching `LINGER_MODEL` to `.env`, then restart the kernel. |
| A name such as `runner` or `case` is undefined | A previous cell was skipped. Use **Restart Kernel and Run All Cells**, or run from the top in order. |
| The first local search takes a long time | FastEmbed may be downloading and initializing the embedding and reranking models. Later searches should be faster. |
| `kind` is `failure` | Read `error_code`. Retry only if `retryable` is `true`; otherwise check setup and scope. |
| The optional all-case run is expensive or slow | Leave `RUN_ALL_CASES = False` and use one selected case instead. |

For the reproducible aggregate retrieval benchmark, run `uv run python -m evals.librarian.benchmark`. For the larger Librarian → Muse → Provenance release evaluation, run `uv run python -m evals.librarian.live_validation`.